<a href="https://colab.research.google.com/github/sivamamidi/Context_Engineering/blob/sivanarayana/Tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import openai
import json


In [43]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# The variables are already defined in cell SBYe4eTYuTBN as Python variables.
# load_dotenv() is not needed here as there's no .env file, and the variables are not set as env vars.
# load_dotenv()

client = AzureOpenAI(
    api_key=GPT4O_API_KEY,
    azure_endpoint=GPT4O_ENDPOINT,
    api_version=GPT4O_API_VERSION,
)

response = client.chat.completions.create(
    model=GPT4O_MODEL_NAME,  # In Azure, this is usually your deployment name
    messages=[
        {"role": "system", "content": "You are a helpful tool-calling agent."},
        {"role": "user", "content": "Check machine M-101 and recommend cooling action."},
    ],
)

print(response.choices[0].message.content)

To assist you, I'll first check the machine M-101's status. Let me retrieve the relevant information.  

Initiating the "Check Machine Status" tool now.  

One moment please.  




In [44]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_bmi",
            "description": "Calculate Body Mass Index from weight and height. Call when user asks about BMI.",
            "parameters": {
                "type": "object",
                "properties": {
                    "weight_kg": {
                        "type": "number",
                        "description": "Body weight in kilograms"
                    },
                    "height_m": {
                        "type": "number",
                        "description": "Height in meters"
                    }
                },
                "required": ["weight_kg", "height_m"]
            }
        }
    }
]

In [45]:
# Step 2: Actual function implementation
def calculate_bmi(weight_kg: float, height_m: float) -> dict:
    bmi = weight_kg / (height_m ** 2)
    category = (
        "Underweight" if bmi < 18.5
        else "Normal" if bmi < 25
        else "Overweight" if bmi < 30
        else "Obese"
    )
    return {"bmi": round(bmi, 1), "category": category}

In [46]:
# Step 3: Function dispatch registry
TOOL_REGISTRY = {
    "calculate_bmi": calculate_bmi
}

In [58]:
#step 4 complete tool calling loop

# Step 4: Complete tool-calling loop
def run_agent(user_message: str):
    messages = [{"role": "user", "content": user_message}]
    #print(tools)

    # First LLM call
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"  # LLM decides whether to use tools
    )

    assistant_msg = response.choices[0].message
    print("9"*99)
    print(assistant_msg)
    print("10"*99)

    # Check if LLM wants to call tools
    if assistant_msg.tool_calls:
        # CRITICAL: Include assistant message with tool_calls in history
        messages.append(assistant_msg)

        for tool_call in assistant_msg.tool_calls:
            func_name = tool_call.function.name
            # CRITICAL: arguments is a STRING, must json.loads() it
            func_args = json.loads(tool_call.function.arguments)

            # Dispatch and execute
            if func_name in TOOL_REGISTRY:
                result = TOOL_REGISTRY[func_name](**func_args)
            else:
                result = {"error": f"Unknown tool: {func_name}"}

            # CRITICAL: role must be "tool" (not "user" or "function")
            # CRITICAL: tool_call_id must match
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        # Second LLM call with tool results
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools
        )
        return final_response.choices[0].message.content

    return assistant_msg.content

In [59]:
print(run_agent("I weigh 75kg and I'm 1.78m tall. What's my BMI?"))

999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999
ChatCompletionMessage(content='The formula for calculating BMI (Body Mass Index) is:\n\n\\[\nBMI = \\frac{\\text{weight (kg)}}{\\text{height (m)}^2}\n\\]\n\nGiven:\n- Weight = 75 kg\n- Height = 1.78 m\n\n\\[\nBMI = \\frac{75}{1.78 \\times 1.78} = \\frac{75}{3.1684} \\approx 23.67\n\\]\n\nYour BMI is approximately **23.67**, which is considered a healthy weight according to most BMI classifications.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010
The formula for calculating BMI (Body Mass Index) is:

\[
BMI = \frac{\text{weight (kg)}}{\text{height (m)}^2}
\]

Given:
- Weight = 75 kg
- Height = 1.78 m

\[
BMI = \frac{75}{1.78 \times 1.78} = \fr

In [49]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get current stock price by ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker, e.g., AAPL"}
                },
                "required": ["ticker"]
            }
        }
    }
]

In [50]:
def get_weather(city):
  return {"city":city,"temp_c":22,"condition":"Sunny"}

def get_stock_price(ticker):
  return {"ticker":ticker,"price":198.50,"currency":"USD"}
REGISTRY = {"get_weather": get_weather, "get_stock_price": get_stock_price}

In [52]:
def run_with_parallel_tools(user_msg):
  messages = [{"role":"user","content":user_msg}]
  response = client.chat.completions.create(
      model='gpt-4o',
      messages=messages,
      tools=tools,
      parallel_tool_calls=True #Explicty enable (default is true)
  )
  msg = response.choices[0].message
  if msg.tool_calls:
    messages.append(msg)
    print(f"LLM requested {len(msg.tool_calls)} parallel tool calls:")
    for tc in msg.tool_calls:
      name = tc.function.name
      args = json.loads(tc.function.arguments)
      print(f"--> {name}{args}")
      result = REGISTRY[name](**args)
      # each result must have matching tool call_id
      messages.append({
          "role":"tool",
          "tool_call_id":tc.id,
          "content":json.dumps(result)
      })

    #single second call -- LLM synthesizes all results

    final = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools
    )
    return final.choices[0].message.content
  return msg.content

In [53]:
print(run_with_parallel_tools("What's the weather in Tokyo and the stock price of AAPL?"))

LLM requested 2 parallel tool calls:
--> get_weather{'city': 'Tokyo'}
--> get_stock_price{'ticker': 'AAPL'}
The current weather in Tokyo is sunny with a temperature of 22°C. 

The current stock price of AAPL (Apple Inc.) is $198.50 USD.
